# Decision: does any predictive-heads condition earn its place?

This notebook combines the prediction results of `part_4`, the latent geometry results
of `part_3` and the training cost measured in `part_0_0` into one decision about which,
if any, classification-head objective should be carried forward over `vpu_precision_sweep`.
It performs no model inference and no new aggregation beyond what those three notebooks
already computed: every quantity is read from their persisted result tables (plus the
base full-campaign cache directly, for the raw geometry values those tables were
themselves derived from), so the decision is reproducible from stored records and
cannot silently diverge from the analyses it summarizes.

## Introduction

Three axes bear on the decision, and a condition that wins on one while losing on
another is not a candidate:

1. **Prediction.** Does the head rank molecules better than `vpu_precision_sweep`, on
   held-out pixels, paired by realized seed?
2. **Representation.** Does the latent keep its variation — a prediction gain bought by
   collapsing the representation is not a gain worth having?
3. **Cost.** How much wall time does the condition add per epoch?

### Assumptions

- **Eight swept conditions are compared against one baseline without a
  multiple-comparison correction.** A single condition whose interval barely excludes
  zero is weak evidence; the pattern within a loss family is what supports a decision.
- **Decision thresholds are stated before the tables are read**, fixed in the
  configuration cell below rather than chosen after seeing which conditions would pass.
- **No "two arms on a common axis" section exists here**, unlike the contractive
  notebook this is ported from: that section compared two *input geometries* the
  contractive penalty could be measured in, a concept with no predictive analogue —
  there is no shared continuous axis these 8 discrete loss families sit on.
- **Angular displacement (encoder sensitivity) is not one of the three gating
  criteria**, mirroring the contractive decision table it is ported from: it is
  informative context (`part_3` computes and plots it), not a pass/fail threshold.

### Notation

For condition $c$ and metric $m$, $\delta_m(c)$ is the paired mean difference against
`vpu_precision_sweep` and $[\ell_m(c), h_m(c)]$ its 95% interval. A condition *separates* on
$m$ when that interval excludes zero. $\mathrm{tr}(u_c)$ is its mean canonicalized
latent trace over the evaluation split.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current_path = Path.cwd().resolve()
repository_root = next(path for path in (current_path, *current_path.parents) if (path / "pyproject.toml").is_file())
os.chdir(repository_root)

from msi_autoencoder_wrapper.analysis.autoencoder.experiments import predictive_campaign as campaign
from msi_autoencoder_wrapper.analysis.autoencoder.experiments import predictive_precompute as cache
from msi_autoencoder_wrapper.analysis.autoencoder.experiments import predictive_reports as reports
from msi_autoencoder_wrapper.utils.logger import get_custom_logger
from msi_autoencoder_wrapper.visualization import resolve_theme
from msi_autoencoder_wrapper.visualization.metrics import plot_violin_with_points

logger = get_custom_logger("decision_summary")

from msi_autoencoder_wrapper.analysis.precompute.notebook_inputs import (
    load_model_catalog, load_visualization_theme, select_catalog_frame,
)


## Configuration and stored inputs

In [ ]:
SETTINGS_PATH = Path("assets/experiments/autoencoder_architecture/notebooks/segmentation_model/17_09_26_metaspace_heads_vpu_contractive/analysis_settings.yaml")
NOTEBOOK_DIR = SETTINGS_PATH.parent
DYNAMICS_RESULTS = NOTEBOOK_DIR / "part_0_0_campaign_training_dynamics_results"
PREDICTION_RESULTS = NOTEBOOK_DIR / "part_4_prediction_global_results"
ARTIFACT_DIR = NOTEBOOK_DIR / "part_7_decision_summary_geometry_vs_prediction_results"
ARTIFACT_DIR.mkdir(exist_ok=True)

# Decision criteria, fixed before any table below is read.
PRIMARY_METRIC = "average_precision"
BASELINE_ROLE = "baseline"
MAX_TRACE_LOSS_FRACTION = 0.20      # a condition may not lose more than this share of latent variation
MAX_COST_MULTIPLE = 2.5             # a condition may not cost more than this multiple of baseline epoch time

settings = campaign.load_settings(SETTINGS_PATH)
from msi_autoencoder_wrapper.analysis.precompute.notebook_inputs import (
    load_model_catalog, load_visualization_theme, select_catalog_frame,
)
models = load_model_catalog(settings)
ready = models[models.ready]
CONDITION_ORDER = reports.condition_order(ready)
baseline_label = ready.query("role == @BASELINE_ROLE").label.iloc[0]

theme = load_visualization_theme(settings)
theme.apply()
LABEL_COLOR = {label: theme.color_for_model(label, index) for index, label in enumerate(CONDITION_ORDER)}

# --- embedded figure resolution --------------------------------------------
FIGURE_RASTER_DPI = 100
plt.rcParams["savefig.dpi"] = FIGURE_RASTER_DPI
# ---------------------------------------------------------------------------


def violin_panel(frame, *, group_column, order, colors, ax, ylabel, title, column="value", reference=None):
    """Draw one violin per group value present in `frame`, contractive-style."""
    present = [value for value in order if value in set(frame[group_column])]
    for position, value in enumerate(present):
        values = frame.loc[frame[group_column] == value, column].dropna().to_numpy()
        if not len(values):
            continue
        plot_violin_with_points(values, position=position, ax=ax, width=0.7, color=colors[value],
                                point_size=10.0, jitter=0.10, theme=theme)
    if reference is not None and np.isfinite(reference):
        ax.axhline(reference, color="grey", linestyle="--", linewidth=1)
    ax.set_xticks(range(len(present)))
    ax.set_xticklabels(present, rotation=45, ha="right", fontsize=7)
    ax.set(ylabel=ylabel, title=title)
    ax.grid(theme.grid_visible, alpha=theme.grid_alpha)

# `run_durations`/`baseline_contrasts` are sibling notebooks' own derived tables, read
# directly like contractive's decision summary reads its three upstream notebooks;
# `prediction`/`geometry` are read straight from the base cache, since predictive's
# per-model tables live in one shared place rather than one silo per notebook.
run_durations = select_catalog_frame(pd.read_csv(DYNAMICS_RESULTS / "run_durations.csv"), models)
prediction_baseline_contrasts = select_catalog_frame(pd.read_csv(PREDICTION_RESULTS / "baseline_contrasts.csv"), models)
prediction = select_catalog_frame(cache.load_table(settings, "prediction"), models)
geometry_frame = select_catalog_frame(cache.load_table(settings, "geometry"), models)

print(f"{len(CONDITION_ORDER)} condition(s); baseline is '{baseline_label}'")

## The three decision quantities as distributions

### Methodology

#### Theoretical

A decision taken on point estimates cannot distinguish a consistent effect from one
divergent seed. Each decision quantity is therefore shown as its distribution over
repetitions before any threshold is applied: ranking quality (average precision on
held-out pixels), retained latent variation ($\operatorname{tr} u$), and wall-clock
cost (mean epoch seconds).

#### Implementation

All three are read from the upstream tables and grouped by condition without
recomputation. The baseline appears as its own condition in every panel.

#### Figure descriptions

Three stacked panels sharing one x-axis of conditions (baseline first). y-axis of the
first panel: validation average precision (`train_supported`, `annotation_retrieval`).
Second: canonicalized latent trace, evaluation split. Third: mean epoch duration in
seconds. Each violin is the distribution over repetitions; the dashed line is the
baseline's mean. Higher is better in the first two panels; lower is cheaper in the
third.

### Remarks

### Notes

In [ ]:
panels = [
    ("validation average precision", prediction.query("split == 'validation' and population == 'annotation_retrieval' and scope == 'train_supported' and metric == @PRIMARY_METRIC"), "value"),
    ("latent trace (u, test)", geometry_frame.query("space == 'u' and split == 'test' and metric == 'trace'"), "value"),
    ("mean epoch duration (s)", run_durations, "mean_epoch_duration"),
]

figure, axes = plt.subplots(len(panels), 1, figsize=(1.3 * len(CONDITION_ORDER) + 3.0, 4.6 * len(panels)), dpi=theme.figure_dpi, sharex=True)

for axis, (title, frame, column) in zip(np.atleast_1d(axes), panels):
    reference = frame.loc[frame.label == baseline_label, column].mean() if baseline_label in set(frame.label) else None
    violin_panel(frame, group_column="label", order=CONDITION_ORDER, colors=LABEL_COLOR, ax=axis, ylabel=title, title=title, column=column, reference=reference)

for axis in np.atleast_1d(axes)[:-1]:
    axis.tick_params(axis="x", labelbottom=False)

np.atleast_1d(axes)[-1].set_xticks(range(len(CONDITION_ORDER)))
np.atleast_1d(axes)[-1].set_xticklabels(CONDITION_ORDER, rotation=45, ha="right", fontsize=7)

figure.suptitle("The three decision quantities, every repetition drawn (dashed line: baseline mean)", y=0.995, fontsize=13)
figure.tight_layout(rect=(0, 0, 1, 0.98))

## Prediction against geometry

### Methodology

#### Theoretical

If a loss choice changes prediction quality through the representation, that mechanism
should be visible as a relation between prediction metrics and geometry statistics
across conditions. Three geometry statistics are used: trace (how much variation
survives), effective rank and participation ratio (how it is spread over directions).
Three prediction metrics are used (`average_precision`, `roc_auc`,
`ap_above_prevalence`) since they can move independently across loss families with very
different objectives.

#### Implementation

Condition means over repetitions are joined across the base `prediction` and
`geometry` tables.

#### Figure descriptions

Grid of scatter panels; x-axis the named geometry statistic (evaluation split, space
`u`), y-axis the named prediction metric (validation split, `annotation_retrieval`,
`train_supported`), one point per condition, the baseline drawn as a black cross.

### Remarks

### Notes

In [ ]:
prediction_means = (
    prediction.query("split == 'validation' and population == 'annotation_retrieval' and scope == 'train_supported'")
    .pivot_table(index="label", columns="metric", values="value", aggfunc="mean")
)
geometry_means = (
    geometry_frame.query("space == 'u' and split == 'test'")
    .pivot_table(index="label", columns="metric", values="value", aggfunc="mean")
)
joined = prediction_means.join(geometry_means, how="inner")
joined_swept = joined.drop(index=baseline_label, errors="ignore").reset_index()

PREDICTION_AXES = ["average_precision", "roc_auc", "ap_above_prevalence"]
GEOMETRY_AXES = ["trace", "effective_rank", "participation_ratio"]

figure, axes = plt.subplots(len(PREDICTION_AXES), len(GEOMETRY_AXES),
                            figsize=(4.2 * len(GEOMETRY_AXES), 3.8 * len(PREDICTION_AXES)),
                            dpi=theme.figure_dpi)
for row, prediction_metric in enumerate(PREDICTION_AXES):
    for column, geometry_metric in enumerate(GEOMETRY_AXES):
        axis = axes[row, column]
        # Every swept condition drawn with its own family colour; the baseline is then
        # overlaid manually as a black cross, matching contractive's own convention for
        # marking the reference cell.
        panel = joined_swept.dropna(subset=[geometry_metric, prediction_metric])
        axis.scatter(panel[geometry_metric], panel[prediction_metric],
                    color=[LABEL_COLOR[label] for label in panel["label"]], s=38, edgecolor="none")
        if baseline_label in joined.index and pd.notna(joined.loc[baseline_label, [geometry_metric, prediction_metric]]).all():
            axis.scatter(joined.loc[baseline_label, geometry_metric], joined.loc[baseline_label, prediction_metric],
                        marker="x", color="black", s=70, linewidth=1.6, zorder=5)
        axis.set_xlabel(geometry_metric if row == len(PREDICTION_AXES) - 1 else "", fontsize=8)
        axis.set_ylabel(prediction_metric if column == 0 else "", fontsize=8)
        axis.set_title("")
        axis.tick_params(labelsize=7)
        axis.grid(theme.grid_visible, alpha=theme.grid_alpha)
figure.suptitle("Prediction metrics against latent geometry, one point per condition (x: baseline)", y=1.02, fontsize=13)
figure.tight_layout()

## Decision table

### Methodology

#### Theoretical

A condition is carried forward only if it satisfies all three criteria stated in the
configuration cell:

1. its paired interval on validation average precision against `vpu_precision_sweep` lies
   strictly above zero;
2. it retains at least $1-\text{MAX\_TRACE\_LOSS\_FRACTION}$ of the baseline's
   latent trace, so an apparent prediction gain is not bought by collapsing the
   representation;
3. its mean epoch duration is at most $\text{MAX\_COST\_MULTIPLE}$ times the
   baseline's.

The first criterion is the substantive one; the other two are guards. All three are
reported per condition whether or not it passes, so a near miss is visible rather than
hidden behind a boolean.

#### Implementation

The paired contrast comes from `part_4`'s persisted `baseline_contrasts.csv`; trace
from the base `geometry` table; cost from `part_0_0`'s persisted `run_durations.csv`.
No quantity is recomputed here.

#### Figure descriptions

Table: one row per swept condition with the paired difference in average precision and
its interval, the retained trace fraction, the cost multiple, the three criterion flags
and the overall verdict. Ordered by paired difference, best first.

### Remarks

### Notes

In [ ]:
baseline_trace = float(geometry_means.loc[baseline_label, "trace"])
baseline_epoch_seconds = float(run_durations.loc[run_durations.label == baseline_label, "mean_epoch_duration"].mean())
epoch_seconds_by_label = run_durations.groupby("label")["mean_epoch_duration"].mean()
ap_contrasts = prediction_baseline_contrasts.query("metric == @PRIMARY_METRIC")

decision_rows = []
for row in ap_contrasts.itertuples():
    label = row.label
    trace = float(geometry_means.loc[label, "trace"]) if label in geometry_means.index else np.nan
    epoch_seconds = epoch_seconds_by_label.get(label, np.nan)
    retained = trace / baseline_trace
    cost_multiple = epoch_seconds / baseline_epoch_seconds
    beats_baseline = bool(np.isfinite(row.ci_low) and row.ci_low > 0)
    keeps_variation = bool(np.isfinite(retained) and retained >= 1.0 - MAX_TRACE_LOSS_FRACTION)
    affordable = bool(np.isfinite(cost_multiple) and cost_multiple <= MAX_COST_MULTIPLE)
    decision_rows.append({"label": label, "mean_difference": row.mean_difference, "ci_low": row.ci_low,
                          "ci_high": row.ci_high, "retained_trace_fraction": retained, "cost_multiple": cost_multiple,
                          "beats_baseline": beats_baseline, "keeps_variation": keeps_variation,
                          "affordable": affordable,
                          "carried_forward": beats_baseline and keeps_variation and affordable})
decision_frame = pd.DataFrame(decision_rows).sort_values("mean_difference", ascending=False)
display(decision_frame.round(4))

## Verdict

### Methodology

#### Implementation

The verdict restates the decision table's outcome and, when no condition passes,
identifies the closest miss on each criterion. The best condition by paired difference
is reported whether or not it passes, since "the best available option is still not
better than the baseline" is itself the decision.

### Remarks

### Notes

In [ ]:
winners = decision_frame[decision_frame["carried_forward"]]
best = decision_frame.iloc[0] if len(decision_frame) else None
print(f"conditions passing every criterion: {len(winners)}")
if len(winners):
    display(winners[["label", "mean_difference", "ci_low", "ci_high", "retained_trace_fraction", "cost_multiple"]].round(4))
else:
    print("No condition satisfies all three criteria.")
    for criterion in ("beats_baseline", "keeps_variation", "affordable"):
        failing = decision_frame[~decision_frame[criterion]]
        print(f"  fails {criterion}: {len(failing)} of {len(decision_frame)} condition(s)")

if best is not None:
    print()
    print(f"best condition by paired difference: {best.label}")
    print(f"  difference {best.mean_difference:+.4f}, interval [{best.ci_low:+.4f}, {best.ci_high:+.4f}]")
    print(f"  retained trace {best.retained_trace_fraction:.3f} of baseline, cost {best.cost_multiple:.2f}x baseline")

separated = decision_frame[decision_frame["beats_baseline"]]
print()
print(f"conditions whose average-precision interval excludes zero on the positive side: {len(separated)}")
if len(separated):
    print("  " + "; ".join(separated["label"]))

## Reproducible artifacts

### Methodology

#### Implementation

The decision table and the joined condition-level view of prediction and geometry are
persisted, together with the criteria that produced the verdict, so the decision can be
audited and re-derived under different thresholds without rerunning any analysis.

### Remarks

### Notes

In [ ]:
decision_frame.to_csv(ARTIFACT_DIR / "decision_table.csv", index=False)
joined.reset_index().to_csv(ARTIFACT_DIR / "condition_level_summary.csv", index=False)

metadata = {
    "primary_metric": PRIMARY_METRIC, "max_trace_loss_fraction": MAX_TRACE_LOSS_FRACTION,
    "max_cost_multiple": MAX_COST_MULTIPLE, "baseline_label": baseline_label,
    "baseline_trace": baseline_trace, "baseline_epoch_seconds": baseline_epoch_seconds,
    "conditions": len(decision_frame), "conditions_carried_forward": int(decision_frame["carried_forward"].sum()),
    "upstream": {"prediction_baseline_contrasts": str(PREDICTION_RESULTS / "baseline_contrasts.csv"),
                "run_durations": str(DYNAMICS_RESULTS / "run_durations.csv")},
}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(sorted(path.name for path in ARTIFACT_DIR.iterdir()))

## Results / Summary

### LLM

Not yet run against the real campaign cache. This notebook has not been executed
against `data/kidney_workspace`, and it depends on `part_0_0`, `part_3` and `part_4`
having been run first (`run_durations.csv`/`baseline_contrasts.csv` must exist).
Write this section from the actually observed decision table and verdict before
treating any interpretation here as verified.

### Person